# Topological Network Analysis & Mathematical Modeling
**Master's Thesis:** *Advertising and Influence Analysis via LLM-Generated Graphs from YouTube*  
**Author:** Aliza Hamid | Universidad Carlos III de Madrid (UC3M)

---
This notebook conducts in-depth mathematical network analysis on the extracted YouTube Knowledge Graph:
- Bipartite Graph Metrics and Degree Distributions
- Weighted Centrality Rankings (Degree, In/Out-Degree, Eigenvector, Betweenness, PageRank)
- Louvain Modularity Clustering
- Independent Cascade Model (ICM) Information Diffusion
- Comparative Evaluation (LLM Extraction vs Baseline Benchmark)

In [ ]:
import json
import math
from pathlib import Path
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

from src.network_analysis import NetworkAnalyzer
from src.kg_visualization import KnowledgeGraphVisualizer

analyzer = NetworkAnalyzer()
G = analyzer.G
print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

## 1. Network Summary and Degree Distribution

In [ ]:
metrics = analyzer.compute_all_metrics()
summary = metrics.get("graph_summary", {})
for k, v in summary.items():
    print(f"{k}: {v}")

# Plot Degree Distribution
degrees = [d for n, d in G.degree()]
plt.figure(figsize=(8, 4.5))
sns.histplot(degrees, bins=15, kde=True, color="#3B82F6")
plt.title("Degree Distribution of YouTube Knowledge Graph Nodes", fontweight="bold")
plt.xlabel("Degree ($k$)")
plt.ylabel("Node Frequency")
plt.show()

## 2. Centrality Metric Rankings & Comparison Table

In [ ]:
node_metrics = metrics.get("node_level_metrics", {})
df_metrics = pd.DataFrame.from_dict(node_metrics, orient="index")
df_metrics.index.name = "entity"
df_metrics.reset_index(inplace=True)

print("--- TOP 10 BRANDS BY IN-DEGREE VIEW WEIGHT ---")
display_cols = ["entity", "node_type", "weighted_degree", "eigenvector", "betweenness", "pagerank"]
top_brands_df = df_metrics[df_metrics["node_type"] == "brand"].sort_values(by="weighted_degree", ascending=False).head(10)
print(top_brands_df[display_cols].to_string(index=False))

print("\n--- TOP CREATORS BY EIGENVECTOR PRESTIGE ---")
top_creators_df = df_metrics[df_metrics["node_type"] == "creator"].sort_values(by="eigenvector", ascending=False)
print(top_creators_df[display_cols].to_string(index=False))

## 3. Community Detection & Sub-Niche Analysis

In [ ]:
comm_res = analyzer.detect_communities()
print(f"Modularity Score: {comm_res.get('modularity_score'):.4f}")

for c in comm_res.get("clusters", []):
    print(f"\nCommunity #{c['community_id']} (Total: {c['size']} nodes)")
    print("  Creators:", c["lead_creators"])
    print("  Brands:", c["lead_brands"][:6])

## 4. Information Diffusion Simulation (ICM Curves)

In [ ]:
diff_res = analyzer.simulate_information_diffusion(propagation_prob=0.35, monte_carlo_trials=50)

plt.figure(figsize=(9, 5))
for strat, res in diff_res.items():
    curve = res.get("cascade_curve", [])
    plt.plot(range(len(curve)), curve, marker="o", label=f"{strat} (Mean Reach: {res['mean_final_reach']:.1f})")

plt.title("Information Diffusion: Cascade Reach over Time Steps", fontweight="bold")
plt.xlabel("Diffusion Step ($t$)")
plt.ylabel("Cumulative Activated Nodes")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.show()

## 5. Baseline Evaluation & Extraction Benchmark

In [ ]:
eval_res = analyzer.benchmark_extraction_fidelity()
print(json.dumps(eval_res, indent=2))